# COVID-19 Tweet Classification Challenge
**Objective**: Predict the sentiment/category of COVID-19 related tweets.
**Metric**: Multi-class Log Loss or Accuracy (depending on exact Zindi version)

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Dataset Class

In [ ]:
class TweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        
        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

## 2. Model Training Function

In [ ]:
def train_epoch(model, data_loader, optimizer, device):
    model.train()
    losses = []
    correct_predictions = 0
    
    for d in tqdm(data_loader):
        input_ids = d["input_ids"].to(device)
        attention_mask = d["attention_mask"].to(device)
        labels = d["labels"].to(device)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        
        loss = outputs.loss
        logits = outputs.logits
        
        _, preds = torch.max(logits, dim=1)
        correct_predictions += torch.sum(preds == labels)
        losses.append(loss.item())
        
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
    return correct_predictions.double() / len(data_loader.dataset), np.mean(losses)

## 3. Main Execution (Example)

In [ ]:
# Load data
try:
    train = pd.read_csv('Train.csv')
    # Assuming columns 'text' and 'target'
    # Encode labels if they are strings
    if train['target'].dtype == 'object':
        from sklearn.preprocessing import LabelEncoder
        le = LabelEncoder()
        train['target'] = le.fit_transform(train['target'])
        num_classes = len(le.classes_)
    else:
        num_classes = train['target'].nunique()

    MODEL_NAME = 'bert-base-uncased'
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    
    train_df, val_df = train_test_split(train, test_size=0.1, random_state=42)
    
    train_ds = TweetDataset(train_df.text.to_numpy(), train_df.target.to_numpy(), tokenizer)
    val_ds = TweetDataset(val_df.text.to_numpy(), val_df.target.to_numpy(), tokenizer)
    
    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)
    
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_classes).to(device)
    optimizer = AdamW(model.parameters(), lr=2e-5, correct_bias=False)
    
    for epoch in range(3):
        train_acc, train_loss = train_epoch(model, train_loader, optimizer, device)
        print(f'Epoch {epoch + 1}: Train loss: {train_loss:.4f}, accuracy: {train_acc:.4f}')
except FileNotFoundError:
    print("Train.csv not found. Please upload the dataset.")